# ESM-2 35M — Full Training-Set Scoring (Colab T4)

> **هذا الـnotebook مكتوب عشان يشتغل على Google Colab بحرف ما يحتاج منك تعديل.**

## ما الهدف؟

نحسب `esm2_llr` لكل ١٩٥ ألف missense variant في مجموعة التدريب + val + test.
هذه الميزة الجديدة راح تُضاف كعمود في الـparquet splits، وبعدين XGBoost يتدرّب مرة ثانية معها.

## الخطوات اللي راح تعملها أنت (٥ دقائق إعداد، ~٣ دقائق يومياً متابعة)

1. افتح هذا النوتبوك في Colab: **File → Open notebook → GitHub → RayanAlDwlah/Genetic-Mutation-Detection-project**
2. **Runtime → Change runtime type → T4 GPU** (مجاناً)
3. **Runtime → Run all** (أو خلية بخلية)
4. لما تطلب خلية 2 إذن الوصول لـDrive، وافق
5. لما تطلب خلية 3 GitHub PAT، الصقه (من Colab → Secrets → اسم السر `GH_PAT`)
6. اتركها تشتغل ~٨ ساعات على T4

## لو انقطعت الجلسة

ما في مشكلة — الـDrive cache يحفظ كل ٥٠٠ variant. **فقط شغّل كل الخلايا مرة ثانية**، راح تكمل من آخر نقطة. لا شي يضيع.

## ١. تثبيت المتطلبات

In [ ]:
# Colab already has torch/transformers; we upgrade + add what's missing.
%pip install -q --upgrade transformers
%pip install -q pyarrow requests

import torch
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## ٢. Mount Google Drive (للـcache)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/missense_esm2/cache'
!mkdir -p {CACHE_DIR}
print('Cache dir:', CACHE_DIR)

## ٣. Clone repo من GitHub

نستخدم الـPAT من Colab Secrets. للإعداد:
1. في Colab: اضغط على 🔑 (يسار)
2. Add new secret
3. Name: `GH_PAT` — Value: توكنك (الذي عنده `repo` + `workflow` scope)
4. فعّل "Notebook access"

In [ ]:
import os
from google.colab import userdata
GH_PAT = userdata.get('GH_PAT')
REPO = 'RayanAlDwlah/Genetic-Mutation-Detection-project'
!rm -rf /content/repo && git clone https://{GH_PAT}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git log --oneline -3

## ٤. تكوين scorer + نسخ الـcache إلى مكان الـrepo

`src/esm2_scorer.py` يعمل resumable cache في `data/intermediate/esm2/`. نربطه بـDrive عن طريق symlink.

In [ ]:
import os
os.makedirs('/content/repo/data/intermediate', exist_ok=True)
# Symlink: data/intermediate/esm2 → Drive cache
local_link = '/content/repo/data/intermediate/esm2'
if os.path.lexists(local_link):
    os.remove(local_link)
os.symlink(CACHE_DIR, local_link)
!ls -la /content/repo/data/intermediate/

## ٥. سكور للـsplits الثلاثة

على T4: ~٠.١٥ ثانية/variant.
- train: ١٩٥K × ٠.١٥ = ~٨ ساعات
- val: ٢٨K × ٠.١٥ = ~١ ساعة
- test: ٢٨K × ٠.١٥ = ~١ ساعة

**المجموع: ~١٠ ساعات**. Colab free tier يسمح ~١٢ ساعة قبل ما يفصل. فيه احتمال يحتاج جلستين.

Cache على Drive محفوظ كل ٥٠٠ variant.

In [ ]:
import sys
sys.path.insert(0, '/content/repo')

import pandas as pd
from src.esm2_scorer import score_variants

for split_name in ['val', 'test', 'train']:
    print(f'\n{"=" * 60}\n[{split_name}] scoring…\n{"=" * 60}')
    df = pd.read_parquet(f'/content/repo/data/splits/{split_name}.parquet')
    query = df[['variant_key', 'chr', 'pos', 'ref', 'alt']].drop_duplicates('variant_key')
    print(f'  {len(query):,} unique variants to score')
    scored = score_variants(query)
    out_path = f'/content/repo/data/intermediate/esm2/scores_{split_name}.parquet'
    scored.to_parquet(out_path, index=False)
    print(f'  [done] {split_name}: {scored["esm2_llr"].notna().sum():,} / {len(scored):,} scored')

print('\n\n✅ ESM-2 scoring complete.')

## ٦. Push النتايج على GitHub

هذه الخلية تأخذ الـparquets الجديدة وتدفعها على `main` مباشرة. استخدام GH_PAT اللي عنده `repo` + `workflow` scopes.

In [ ]:
%cd /content/repo
!git config user.email 'colab@automation'
!git config user.name 'Colab Automation'
# Only commit the score parquets (they're symlinked to Drive so git sees them as regular files).
!cp /content/drive/MyDrive/missense_esm2/cache/scores_*.parquet data/intermediate/esm2/
!git add data/intermediate/esm2/scores_train.parquet data/intermediate/esm2/scores_val.parquet data/intermediate/esm2/scores_test.parquet
!git commit -m 'Stage 2.1 ESM-2: full-training-set zero-shot LLR scoring (Colab T4)'
!git push origin HEAD:main

## ٧. التالي

الآن على جهازي المحلي، أنا راح أسحب الـparquets الجديدة وأعيد تدريب XGBoost مع العمود الجديد `esm2_llr`. النتيجة راح تنزل تلقائياً على `main`.

### لو طلعت مشكلة
- **خطأ CUDA**: تأكد من Runtime → T4 GPU
- **خطأ Drive**: اعمل re-authorize في الخلية ٢
- **خطأ VEP REST**: لو طلع `502` أو `503`، انتظر ١٠ دقائق ثم شغّل مرة ثانية
- **انقطاع الجلسة**: ما فيه مشكلة — فقط اعد تشغيل كل الخلايا، الـcache في Drive محفوظ

أي مشكلة ثانية أرسلها لي.